<a href="https://colab.research.google.com/github/Shanmuganathan75/QM640-WALSH-CAPSTONE/blob/main/03_build_screening_worksheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QM 640 Capstone — Step 3: Manual Screening & Announcement-Type Classification

This step is deliberately NOT automated end-to-end — the Synopsis's Data
Quality Risk section requires manual review of 8-K text and newswire
language to (a) apply the exclusion criteria and (b) classify
`announcement_type`. This notebook builds the worksheet and, later, checks
inter-rater reliability. **The actual screening happens outside this
notebook, in a spreadsheet.**

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every
other cell in this notebook reads from and writes to.

In [ ]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "Shan_muganathan@yahoo.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 43 (delta 15), reused 39 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 406.29 KiB | 8.64 MiB/s, done.
Resolving deltas: 100% (15/15), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [ ]:
!pip install -q pandas scikit-learn

## Cell 3 — Configuration

In [ ]:
import os

RAW_DIR = os.path.join(BASE_DIR, "data/raw")
SCREENING_FILE = os.path.join(RAW_DIR, "screening_worksheet.csv")
RECODE_FILE = os.path.join(RAW_DIR, "screening_recode_sample.csv")

## Part A — Build the worksheet (run once)

Converts the raw EDGAR candidate pool into a screening worksheet with blank
columns for manual review, plus a 20% random subsample set aside for
independent re-coding.

In [ ]:
import pandas as pd


def build_worksheet():
    src = os.path.join(RAW_DIR, "edgar_candidate_events.csv")
    df = pd.read_csv(src)

    df["file_date"] = pd.to_datetime(df["file_date"])
    df = df.sort_values("file_date").drop_duplicates(subset=["cik", "file_date"], keep="first")

    df["is_genuine_ai_event"] = ""       # Y/N - excludes incidental AI mentions
    df["announcement_type"] = ""          # partnership / R&D / M&A
    df["confounding_event_flag"] = ""     # Y/N - other material event in [-2,+2]
    df["trading_halt_flag"] = ""          # Y/N
    df["sufficient_history_flag"] = ""    # Y/N - >=120 trading days pre-event
    df["exclude_reason"] = ""             # free text if excluded
    df["filing_url"] = df.apply(
        lambda r: f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={r['cik']}",
        axis=1,
    )
    df["screener_notes"] = ""

    df.to_csv(SCREENING_FILE, index=False)
    print(f"Worksheet built: {len(df)} candidate events -> {SCREENING_FILE}")

    n_subsample = max(1, int(len(df) * 0.20))
    subsample = df.sample(n=n_subsample, random_state=42)
    subsample_out = subsample[["accession_no", "company_name", "file_date"]].copy()
    subsample_out["recoder_announcement_type"] = ""
    subsample_out["recoder_is_genuine_ai_event"] = ""
    subsample_out.to_csv(RECODE_FILE, index=False)
    print(f"20% re-coding sample ({n_subsample} events) -> {RECODE_FILE}")
    return df


worksheet = build_worksheet()
worksheet.head()

Worksheet built: 2496 candidate events -> /content/QM640-WALSH-CAPSTONE/data/raw/screening_worksheet.csv
20% re-coding sample (499 events) -> /content/QM640-WALSH-CAPSTONE/data/raw/screening_recode_sample.csv


,query,cik,company_name,form_type,file_date,accession_no,adsh,file_name,is_genuine_ai_event,announcement_type,confounding_event_flag,trading_halt_flag,sufficient_history_flag,exclude_reason,filing_url,screener_notes
2218,"""machine learning investment""",1144879,"Applied Blockchain, Inc. (APLD) (CIK 0001144...",8-K,2023-01-10,0001144879-23-000026:apldjanuarypresentation1.htm,0001144879-23-000026,NaN,,,,,,,https://www.sec.gov/cgi-bin/browse-edgar?actio...,
2837,"""large language model""",1822145,"Presto Automation Inc. (PRST, PRSTW) (CIK 00...",8-K,2023-01-11,0001558370-23-000205:prst-20230110xex99d1.htm,0001558370-23-000205,NaN,,,,,,,https://www.sec.gov/cgi-bin/browse-edgar?actio...,
934,"""generative AI""",1549346,"Shutterstock, Inc. (SSTK) (CIK 0001549346)",8-K,2023-02-09,0001549346-23-000008:a2022-q4_exx991xpressrele...,0001549346-23-000008,NaN,,,,,,,https://www.sec.gov/cgi-bin/browse-edgar?actio...,
1184,"""generative AI""",1066194,EGAIN Corp (EGAN) (CIK 0001066194),8-K,2023-02-14,0001558370-23-001272:egan-20230214xex99d1.htm,0001558370-23-001272,NaN,,,,,,,https://www.sec.gov/cgi-bin/browse-edgar?actio...,
1005,"""generative AI""",1627475,"UPWORK, INC (UPWK) (CIK 0001627475)",8-K,2023-02-15,0001627475-23-000004:final_q4fy22upworkshareh.htm,0001627475-23-000004,NaN,,,,,,,https://www.sec.gov/cgi-bin/browse-edgar?actio...,


## Commit and push results back to GitHub

In [ ]:
!git -C {BASE_DIR} add "data/raw/screening_worksheet.csv"
!git -C {BASE_DIR} add "data/raw/screening_recode_sample.csv"
!git -C {BASE_DIR} commit -m "Step 3a: build screening worksheet + recode sample"
!git -C {BASE_DIR} push

[main 0d31fcd] Step 3a: build screening worksheet + recode sample
 2 files changed, 2997 insertions(+), 2697 deletions(-)
 rewrite data/raw/screening_recode_sample.csv (81%)
 rewrite data/raw/screening_worksheet.csv (71%)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (6/6), 55.53 KiB | 1.32 MiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 2 local objects.
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   ecbb1f1..0d31fcd  main -> main


## >>> STOP HERE — manual step, outside this notebook <<<

1. Download `screening_worksheet.csv` (or edit it directly on GitHub / in
   Google Sheets after downloading) and manually review each row against
   the exclusion criteria in the Synopsis, using the `filing_url` link to
   read the actual 8-K text.
2. Fill in `is_genuine_ai_event`, `announcement_type`,
   `confounding_event_flag`, `trading_halt_flag`, `sufficient_history_flag`.
3. Re-upload the completed CSV back into `data/raw/screening_worksheet.csv`
   in the repo (via `git push` from your local machine, GitHub's web upload,
   or by re-running Cell 1 in a fresh Colab session, replacing the file, and
   running the push cell below).
4. Give `screening_recode_sample.csv` to an independent reviewer, **without
   your own classifications visible**, and have them fill in
   `recoder_announcement_type` / `recoder_is_genuine_ai_event`. Merge that
   back into the repo the same way.

Once both files are updated in the repo, continue to Part B below.

## Part B — Cohen's kappa on the re-coded sample

Run this after the manual screening and independent re-coding are both
complete and pushed to the repo. Re-run Cell 1 first if this is a new
session, to pull the latest screened data.

In [ ]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"
REPO_NAME = "QM640-WALSH-CAPSTONE"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(f"Clone failed: no .git folder found at {BASE_DIR}")

print("Repo ready at:", BASE_DIR)
!ls {BASE_DIR}/data/raw

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 96, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 96 (delta 35), reused 77 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (96/96), 886.15 KiB | 10.07 MiB/s, done.
Resolving deltas: 100% (35/35), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE
cik_ticker_map.csv	    screening_recode_sample.csv  sp500_constituents.csv
edgar_candidate_events.csv  screening_TO_REVIEW.csv
firm_size.csv		    screening_worksheet.csv


In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score
import os
from google.colab import userdata # Added for BASE_DIR dependency

# --- Start of BASE_DIR and related definitions (usually from Cell 419ed1b5) ---
# These definitions are included here to ensure the cell runs independently,
# in case the setup cell (Cell 419ed1b5) was not executed or the kernel was reset.
GITHUB_USERNAME = "Shanmuganathan75"
REPO_NAME = "QM640-WALSH-CAPSTONE"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
BASE_DIR = f"/content/{REPO_NAME}"
# --- End of BASE_DIR and related definitions ---

# Definitions moved from cell 14ac5461 to ensure they are in scope.
RAW_DIR = os.path.join(BASE_DIR, "data/raw")
SCREENING_FILE = os.path.join(RAW_DIR, "screening_worksheet.csv")
RECODE_FILE = os.path.join(RAW_DIR, "screening_recode_sample.csv")

# --- Start of build_worksheet function (moved from Cell fea5bf9f) ---
def build_worksheet():
    src = os.path.join(RAW_DIR, "edgar_candidate_events.csv")
    # Check if edgar_candidate_events.csv exists before proceeding
    if not os.path.exists(src):
        print(f"Error: Required source file '{src}' not found. Please ensure it exists.")
        return pd.DataFrame() # Return empty DataFrame to prevent further errors

    df = pd.read_csv(src)

    df["file_date"] = pd.to_datetime(df["file_date"])
    df = df.sort_values("file_date").drop_duplicates(subset=["cik", "file_date"], keep="first")

    df["is_genuine_ai_event"] = ""       # Y/N - excludes incidental AI mentions
    df["announcement_type"] = ""          # partnership / R&D / M&A
    df["confounding_event_flag"] = ""     # Y/N - other material event in [-2,+2]
    df["trading_halt_flag"] = ""          # Y/N
    df["sufficient_history_flag"] = ""    # Y/N - >=120 trading days pre-event
    df["exclude_reason"] = ""             # free text if excluded
    df["filing_url"] = df.apply(
        lambda r: f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={r['cik']}",
        axis=1,
    )
    df["screener_notes"] = ""

    df.to_csv(SCREENING_FILE, index=False)
    print(f"Worksheet built: {len(df)} candidate events -> {SCREENING_FILE}")

    n_subsample = max(1, int(len(df) * 0.20))
    subsample = df.sample(n=n_subsample, random_state=42)
    subsample_out = subsample[["accession_no", "company_name", "file_date"]].copy()
    subsample_out["recoder_announcement_type"] = ""
    subsample_out["recoder_is_genuine_ai_event"] = ""
    subsample_out.to_csv(RECODE_FILE, index=False)
    print(f"20% re-coding sample ({n_subsample} events) -> {RECODE_FILE}")
    return df
# --- End of build_worksheet function ---


def calculate_kappa():
    # Ensure the screening files exist; if not, build them.
    if not os.path.exists(SCREENING_FILE):
        print(f"'{SCREENING_FILE}' not found. Rebuilding worksheets...")
        build_worksheet()
        # After building, check again if they exist. If not, something is fundamentally wrong.
        if not os.path.exists(SCREENING_FILE):
            print("Failed to build worksheets. Cannot proceed with kappa calculation.")
            return None

    original = pd.read_csv(SCREENING_FILE)
    recode = pd.read_csv(RECODE_FILE)

    merged = recode.merge(
        original[["accession_no", "announcement_type"]], on="accession_no", how="left"
    )
    merged = merged.dropna(subset=["announcement_type", "recoder_announcement_type"])
    merged = merged[merged["recoder_announcement_type"] != ""]

    if len(merged) < 2:
        print("Not enough re-coded rows yet. Fill in recoder_announcement_type first.")
        return None

    kappa = cohen_kappa_score(merged["announcement_type"], merged["recoder_announcement_type"])
    print(f"Cohen's kappa (announcement_type, n={len(merged)}): {kappa:.3f}")

    if kappa < 0.70:
        print("\nKAPPA BELOW .70 - per the Synopsis, this triggers a joint review "
              "of the classification criteria before finalizing the full sample.")
        disagreements = merged[merged["announcement_type"] != merged["recoder_announcement_type"]]
        print(disagreements[["company_name", "announcement_type", "recoder_announcement_type"]])
    else:
        print("Kappa meets the .70 threshold - classification is reliable enough to proceed.")
    return kappa


kappa_result = calculate_kappa()

Cohen's kappa (announcement_type, n=4): 1.000
Kappa meets the .70 threshold - classification is reliable enough to proceed.
